# Ablation Study: Hosseini FAST & RIGOROUS (Robust Split Fix)

## Overview
1. **Robust Split**: Split percentuale (70/15/15) che si adatta al numero di soggetti effettivamente trovati.
2. **Weighted Loss**: Manteniamo il bilanciamento per evitare il bias della classe maggioritaria.
3. **Safe Dataloaders**: Controllo che i set non siano vuoti.

In [ ]:
import os, sys, shutil, glob, pickle, h5py, re, copy, random
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, pandas as pd
from einops import rearrange
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

# ── Reproducibility seed (aggiunto per riproducibilità) ────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

WORK_DIR = '/kaggle/working/e4selflearning'
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}
!pip install -q einops scikit-learn
if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)
print("Setup complete. Seed:", SEED)

In [ ]:
ABLATION_TAGS = ['only_bigideas', 'only_spd', 'only_ue4w', 'only_weee', 'only_wesd', 'bigideas_spd', 'bigideas_spd_ue4w', 'bigideas_spd_ue4w_weee', 'all_datasets']
CHANNELS  = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
MAX_EPOCHS = 50
PATIENCE = 7
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ── ARCHITECTURES ──────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size, d_model):
        super().__init__()
        self.proj = nn.Linear(patch_size, d_model); self.norm = nn.LayerNorm(d_model)
    def forward(self, x): return self.norm(self.proj(x))

class SimMTMEncoder(nn.Module):
    def __init__(self, seg_len=640, patch_size=16, d_model=128, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.patch_size, self.d_model, self.n_patches = patch_size, d_model, seg_len // patch_size
        self.patch_emb = PatchEmbedding(patch_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers); self.norm = nn.LayerNorm(d_model)
    def patchify(self, x):
        return rearrange(x[:, :, :self.n_patches * self.patch_size], 'b c (n p) -> (b c) n p', p=self.patch_size)
    def forward(self, x):
        return self.norm(self.transformer(self.patch_emb(self.patchify(x)) + self.pos_emb))

class SimMTMClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder; self.head = nn.Linear(encoder.d_model, 1)
    def forward(self, x):
        feats = self.encoder(x)
        feats = rearrange(feats, '(b c) n d -> b c n d', c=6)
        return self.head(feats.mean(dim=(1, 2)))

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model); pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class BIOTEncoder(nn.Module):
    def __init__(self, n_channels=6, seg_len=640, chunk_size=64, d_model=256, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.n_channels, self.chunk_size, self.d_model, self.n_chunks = n_channels, chunk_size, d_model, seg_len // chunk_size
        self.patch_embed = nn.Linear(chunk_size, d_model); self.channel_emb = nn.Embedding(n_channels, d_model)
        self.pos_enc = SinusoidalPE(d_model, max_len=self.n_chunks * n_channels + 1); self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers); self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        B, C, T = x.shape
        x = rearrange(x[:, :, :self.n_chunks * self.chunk_size], 'b c (n p) -> b c n p', p=self.chunk_size)
        x = self.patch_embed(x) + self.channel_emb(torch.arange(C, device=x.device)).unsqueeze(0).unsqueeze(2)
        x = torch.cat([self.cls_token.expand(B, -1, -1), rearrange(x, 'b c n d -> b (c n) d')], dim=1)
        return self.norm(self.transformer(self.pos_enc(x)))[:, 0, :]

class BIOTClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder; self.head = nn.Linear(encoder.d_model, 1)
    def forward(self, x): return self.head(self.encoder(x))

In [ ]:
# ── DATA PREP (ROBUST SPLIT) ────────────────────────────────────────────────
all_metas = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
src_meta = [m for m in all_metas if 'indianipostfase2' in m.lower()][0]
src_dir = os.path.dirname(src_meta)
with open(src_meta, "rb") as f: meta = pickle.load(f)

label_key = "status" if "status" in meta["sessions_labels"] else "target"
raw_labels = np.array(meta["sessions_labels"][label_key])
paths_all = np.array([os.path.join(src_dir, p.replace("\\", "/").split("/")[-2], p.replace("\\", "/").split("/")[-1]) for p in meta["sessions_paths"]])

binary_labels = raw_labels.astype(float) if all(v <= 1 for v in np.unique(raw_labels)) else np.zeros_like(raw_labels)
if not all(v <= 1 for v in np.unique(raw_labels)):
    binary_labels[np.isin(raw_labels, [1, 2, 3])] = 1

def get_sub(p): return os.path.basename(os.path.dirname(p)).split('_')[0]
all_subs = sorted(list(set([get_sub(p) for p in paths_all])), key=lambda x: int(x) if x.isdigit() else x)
np.random.seed(42); np.random.shuffle(all_subs)

# SPLIT PERCENTUALE ROBUSTO (70/15/15)
n_total = len(all_subs)
tr_n = max(1, int(n_total * 0.7))
vl_n = max(1, int(n_total * 0.15))
train_subs, val_subs, test_subs = all_subs[:tr_n], all_subs[tr_n:tr_n+vl_n], all_subs[tr_n+vl_n:]
if not test_subs and n_total >= 3: # Fail-safe se lo split mangia l'ultimo
    test_subs = [all_subs[-1]]; val_subs = [all_subs[-2]]; train_subs = all_subs[:-2]

print(f"Subjects found: {n_total}. Split: Train={len(train_subs)}, Val={len(val_subs)}, Test={len(test_subs)}")

class FastRAMDataset(Dataset):
    def __init__(self, paths, labels):
        self.data, self.y = [], []
        if len(paths) == 0: return
        print(f"Loading {len(paths)} segments...")
        for p, l in tqdm(zip(paths, labels), total=len(paths)):
            try:
                with h5py.File(p, 'r') as f:
                    sigs = []
                    for c in CHANNELS:
                        s = torch.from_numpy(f[c][()]).float()
                        s = (s - s.mean()) / (s.std() + 1e-8)
                        sigs.append(F.interpolate(s.view(1, 1, -1), size=640, mode='linear', align_corners=False).squeeze())
                    self.data.append(torch.stack(sigs)); self.y.append(l)
            except: continue
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], torch.tensor([self.y[i]]).float()

m_tr = np.array([get_sub(p) in train_subs for p in paths_all])
m_vl = np.array([get_sub(p) in val_subs for p in paths_all])
m_ts = np.array([get_sub(p) in test_subs for p in paths_all])

ds_train, ds_val, ds_test = FastRAMDataset(paths_all[m_tr], binary_labels[m_tr]), FastRAMDataset(paths_all[m_vl], binary_labels[m_vl]), FastRAMDataset(paths_all[m_ts], binary_labels[m_ts])
train_loader = DataLoader(ds_train, batch_size=16, shuffle=True) if len(ds_train)>0 else None
val_loader = DataLoader(ds_val, batch_size=16, shuffle=False) if len(ds_val)>0 else None
test_loader = DataLoader(ds_test, batch_size=16, shuffle=False) if len(ds_test)>0 else None

if train_loader:
    train_y = np.array([d[1].item() for d in ds_train])
    pos_weight = torch.tensor([(len(train_y) - sum(train_y)) / (sum(train_y) + 1e-8)]).to(device)
    print(f"\nTrain Class Ratio: {np.mean(train_y):.2f}, Pos Weight: {pos_weight.item():.2f}")
else: pos_weight = torch.tensor([1.0]).to(device)

if test_loader: print(f"Test Set Binary Distribution: {np.unique([d[1].item() for d in ds_test], return_counts=True)}")
else: print("\nWARNING: Test Set is EMPTY!")

In [ ]:
def fine_tune_and_evaluate(model, train_loader, val_loader, test_loader, device, pos_weight):
    if train_loader is None or test_loader is None: return 0.0, 0.0, 0.0
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_val_loss = float('inf'); best_model = None; counter = 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for x, y in train_loader:
            opt.zero_grad(); out = model(x.to(device)).reshape(-1)
            loss = crit(out, y.to(device).reshape(-1)); loss.backward(); opt.step()
        model.eval(); val_loss = 0
        if val_loader:
            with torch.no_grad():
                for x, y in val_loader:
                    out = model(x.to(device)).reshape(-1)
                    val_loss += crit(out, y.to(device).reshape(-1)).item()
            if val_loss < best_val_loss: best_val_loss = val_loss; best_model = copy.deepcopy(model.state_dict()); counter = 0
            else: 
                counter += 1
                if counter >= PATIENCE: break
    if best_model: model.load_state_dict(best_model)
    model.eval(); preds, truths = [], []
    with torch.no_grad():
        for x, y in test_loader:
            out = model(x.to(device)).reshape(-1)
            preds.extend((torch.sigmoid(out) >= 0.5).long().cpu().tolist()); truths.extend(y.reshape(-1).tolist())
    return accuracy_score(truths, preds), f1_score(truths, preds, average='macro'), f1_score(truths, preds, zero_division=0)

results = []
for tag in ABLATION_TAGS:
    print(f"\nEvaluating: {tag}")
    for m_type in ['simmtm', 'biot']:
        filename = f'{m_type}_encoder_ablation_{tag}.pt'
        found = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
        if found:
            pt = found[0]; enc = SimMTMEncoder().to(device) if m_type == 'simmtm' else BIOTEncoder().to(device)
            enc.load_state_dict(torch.load(pt, map_location=device))
            clf = SimMTMClassifier(enc).to(device) if m_type == 'simmtm' else BIOTClassifier(enc).to(device)
            acc, f1m, f1s = fine_tune_and_evaluate(clf, train_loader, val_loader, test_loader, device, pos_weight)
            print(f"  {m_type} -> Acc={acc:.4f} | F1-Stress={f1s:.4f}")
            results.append({'model': m_type, 'tag': tag, 'accuracy': acc, 'f1_macro': f1m, 'f1_stress': f1s})

pd.DataFrame(results).to_csv('/kaggle/working/ablation_results_HOSSEINI_balanced.csv', index=False)
print("\n--- FINISHED ---")